In [ ]:
import os 
import sys 
import warnings
from pyspark.sql.types import (
    StructField, StructType, IntegerType, StringType
)
sys.path.append(os.path.abspath(os.path.join("..")))

from src.spark import get_spark_session
from configs.config import RAW_DATA_DIR, BRONZE_DATA_DIR

warnings.filterwarnings("ignore")

In [ ]:
spark = get_spark_session() # get spark session

spark

## `Bronze Ingestion Contract`
 The bronze layer preserves the source dataset while enforcing the expected structure of the incoming data

 ### `Required fields`
 - rank
 - Youtuber
 - subscribers
 - video views
 - video count
 - category
 - started

 ### `Expectations`
 - "rank" must be an integer and must be not-null
 - "Youtuber" must be a string and not-null
 - "subscribers" must be an integer or parseable string
 - "video views" must be an integer or parseable string
 - "video count" must be an integer or parseable string
 - "category" must be a string
 - "started" must be an integer or parseable string
 - all required columns must exist
 - dataset must contain at least one record

In [12]:
youtube_schema = StructType([ # define schema
    StructField("rank", IntegerType(), nullable=True),
    StructField("Youtuber", StringType(), nullable=True),
    StructField("subscribers", StringType(), nullable=True),
    StructField("video views", StringType(), nullable=True),
    StructField("video count", StringType(), nullable=True),
    StructField("category", StringType(), nullable=True),
    StructField("started", IntegerType(), nullable=True),
])  

`READ DATA FROM CSV FILE INTO SPARK DATAFRAME`

In [ ]:
list(RAW_DATA_DIR.iterdir()) # list files in source folder directory

[PosixPath('/home/mhaxs/germany_youtube_analytics_pipeline/data/raw/top_1000_msycig.csv')]

In [ ]:
source_file = RAW_DATA_DIR / "top_1000_msycig.csv"

source_file

PosixPath('/home/mhaxs/germany_youtube_analytics_pipeline/data/raw/top_1000_msycig.csv')

In [15]:
df_bronze = ( # read data into dataframe from csv file
    spark.read
    .option("header", True)
    .schema(youtube_schema)
    .csv(str(source_file))
)

In [16]:
df_bronze.show()

+----+--------------------+-----------+--------------+-----------+--------------------+-------+
|rank|            Youtuber|subscribers|   video views|video count|            category|started|
+----+--------------------+-----------+--------------+-----------+--------------------+-------+
|   1|        Tsuriki Show| 34,100,000|42,490,526,838|      4,739|       Entertainment|   2019|
|   2|Kidibli (Kinder S...| 29,600,000|15,673,364,837|      1,236|       Entertainment|   2015|
|   3|Kurzgesagt – In a...| 23,600,000| 3,145,706,013|        271|           Education|   2013|
|   4|            boxtoxtv| 23,500,000|18,303,986,629|      1,559|              Comedy|   2022|
|   5|          HaerteTest| 19,500,000| 3,420,864,412|      1,712|Science & Technology|   2011|
|   6|       Noel Robinson| 18,600,000|10,752,582,783|      1,639|       Entertainment|   2015|
|   7|        FAMILY BOOMS| 16,800,000|16,225,259,066|      1,587|       Entertainment|   2021|
|   8|      Talking Angela| 13,300,000| 

In [17]:
df_bronze.printSchema()

root
 |-- rank: integer (nullable = true)
 |-- Youtuber: string (nullable = true)
 |-- subscribers: string (nullable = true)
 |-- video views: string (nullable = true)
 |-- video count: string (nullable = true)
 |-- category: string (nullable = true)
 |-- started: integer (nullable = true)



`DATA QUALITY GATES`

In [18]:
row_counts = df_bronze.count() # CHECK FOR REQUIRED OR PERCEIVED ROW COUNT

if row_counts == 0:
    raise ValueError("Bronze ingestion failed: Dataset is empty")

print("Bronze row count: ", row_counts)

Bronze row count:  1000


In [19]:
expected_columns = set(youtube_schema.fieldNames()) # MATCH ACTUAL COLUMNS AGAINST EXPECTED COLUMNS
actual_columns = set(df_bronze.columns)

missing_columns = expected_columns - actual_columns

if missing_columns:
    raise ValueError(f"Bronze schema validation failed. Missing columns: {missing_columns}")

print("Required columns check: PASS")

Required columns check: PASS


In [20]:
if df_bronze.schema != youtube_schema: # CHECK SCHEMA CORRECTNESS
    raise ValueError(
        "Bronze schema validation failed: Actual schema doees not match expected schema"
    ) # DELIBERATELY STRICT BY HALTING PIPELINE
    # TODO: SCHEMA EVOLUTION
    
print("Schema check: PASS")

Schema check: PASS


In [21]:
bronze_path = BRONZE_DATA_DIR / "youtube_channels"

In [22]:
( # write data to bronze in a parquet file format
    df_bronze.write
    .mode("overwrite")
    .parquet(str(bronze_path))
)

In [23]:
df_bronze = spark.read.parquet(str(bronze_path))

In [24]:
df_bronze.show(10)

+----+--------------------+-----------+--------------+-----------+--------------------+-------+
|rank|            Youtuber|subscribers|   video views|video count|            category|started|
+----+--------------------+-----------+--------------+-----------+--------------------+-------+
|   1|        Tsuriki Show| 34,100,000|42,490,526,838|      4,739|       Entertainment|   2019|
|   2|Kidibli (Kinder S...| 29,600,000|15,673,364,837|      1,236|       Entertainment|   2015|
|   3|Kurzgesagt – In a...| 23,600,000| 3,145,706,013|        271|           Education|   2013|
|   4|            boxtoxtv| 23,500,000|18,303,986,629|      1,559|              Comedy|   2022|
|   5|          HaerteTest| 19,500,000| 3,420,864,412|      1,712|Science & Technology|   2011|
|   6|       Noel Robinson| 18,600,000|10,752,582,783|      1,639|       Entertainment|   2015|
|   7|        FAMILY BOOMS| 16,800,000|16,225,259,066|      1,587|       Entertainment|   2021|
|   8|      Talking Angela| 13,300,000| 

In [25]:
df_bronze.printSchema()

root
 |-- rank: integer (nullable = true)
 |-- Youtuber: string (nullable = true)
 |-- subscribers: string (nullable = true)
 |-- video views: string (nullable = true)
 |-- video count: string (nullable = true)
 |-- category: string (nullable = true)
 |-- started: integer (nullable = true)



In [26]:
df_bronze.count()

1000